In [ ]:
from bing_image_downloader import downloader

# função para baixar imagens
def DownloadPics(query, output_dir, limit):
    print(f'Iniciando download das fotos: {query}')
    downloader.download(query=query,
                        limit=limit,
                        output_dir=output_dir,
                        adult_filter_off=True,
                        force_replace=False,
                        # se limite é menor que 100, timeout é de 60s, se for maior, calcula proporcionalmente (+ 10 para ter certeza)
                        timeout=60)
    
# pesquisa de imagens
query_n = 'Neymar face png'
query_m = 'Messi face png'

# limite de imagens por download
limit = 100

# diretório de saída
output_dir_n = 'data/neymar/'
output_dir_m = 'data/messi/'

# baixar fotos do neymar
DownloadPics(query=query_n, output_dir=output_dir_n, limit=limit)

# baixar fotos do messi
DownloadPics(query=query_m, output_dir=output_dir_m, limit=limit)

Iniciando download das fotos: {input}
[%] Downloading Images to /home/liperasz/LAMIA/bootcamp-machine-learning/Card-21/Pratica/data/neymar/Neymar face png


[!!]Indexing page: 1

[%] Indexed 47 Images on Page 1.


[%] Downloading Image #1 from https://i.pinimg.com/736x/5a/8c/a9/5a8ca90531297273bc47d19a66a1c438.jpg
[%] File Downloaded !

[%] Downloading Image #2 from https://www.kindpng.com/picc/m/108-1088287_neymar-jr-png-face-neymar-barca-png-transparent.png
[%] File Downloaded !

[%] Downloading Image #3 from https://crystalpng.com/wp-content/uploads/2025/03/neymar-png-1024x1024.png
[%] File Downloaded !

[%] Downloading Image #4 from https://icon2.cleanpng.com/20230601/vuu/transparent-white-background-1711131584725.webp
[%] File Downloaded !

[%] Downloading Image #5 from https://www.pngitem.com/pimgs/m/57-571934_neymar-jr-png-face-neymar-en-el-real.png
[%] File Downloaded !

[%] Downloading Image #6 from https://www.clipartmax.com/png/middle/263-2633230_neymar-dribbling-neymar-hd-p

In [2]:
# Baixa o script auxiliar necessário para distorcer as imagens no treino
import os
if not os.path.exists('random_warp.py'):
    !wget -q https://raw.githubusercontent.com/sizhky/deep-fake-util/main/random_warp.py

from torch_snippets import *
from random_warp import get_training_data
import cv2
import torch
import torch.nn as nn
import torch.optim as optim

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
# detector de rostos do opencv
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# função para recortar a imagem
def crop_face(img):

    # imagem em cinza
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # detecta os rostos
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    if(len(faces)>0):
        for (x,y,w,h) in faces:
            img2 = img[y:(y+h),x:(x+w),:]
            # redimensiona a imagem
            img2 = cv2.resize(img2,(256,256))
            return img2, True
    else:
        return img, False

# Cria as pastas para os rostos recortados
!mkdir -p cropped_neymar
!mkdir -p cropped_messi

# separa as imagens
def crop_images(folder, output_folder):
    # pega as imagens
    images = Glob(folder+'/*.*')
    for i in range(len(images)):
        try:
            img = read(images[i],1)
            img2, face_detected = crop_face(img)
            # se detectar o rosto, salva a imagem cropada
            if face_detected:
                cv2.imwrite(output_folder+'/'+str(i)+'.jpg', cv2.cvtColor(img2, cv2.COLOR_RGB2BGR))
        except:
            continue # Pula imagens corrompidas

crop_images('data/neymar', 'cropped_neymar')
crop_images('data/messi', 'cropped_messi')

In [4]:
from torch.utils.data import Dataset, DataLoader
import random


# definindo o dataset
class ImageDataset(Dataset):
    def __init__(self, items_A, items_B):
        # converte as imagens para float e normaliza elas
        self.items_A = np.concatenate([read(f,1)[None] for f in items_A])/255.
        self.items_B = np.concatenate([read(f,1)[None] for f in items_B])/255.
        self.items_A += self.items_B.mean(axis=(0, 1, 2)) - self.items_A.mean(axis=(0, 1, 2))

    def __len__(self):
        return min(len(self.items_A), len(self.items_B))

    def __getitem__(self, ix):
        a, b = random.choice(self.items_A), random.choice(self.items_B)
        return a, b

    def collate_fn(self, batch):
        # converte para tensor e divide em imagem de treino e target
        imsA, imsB = list(zip(*batch))
        imsA, targetA = get_training_data(imsA, len(imsA))
        imsB, targetB = get_training_data(imsB, len(imsB))
        imsA, imsB, targetA, targetB = [torch.Tensor(i).permute(0,3,1,2).to(device) for i in [imsA, imsB, targetA, targetB]]
        return imsA, imsB, targetA, targetB

# instanciando dataset e dataloader
dataset = ImageDataset(Glob('cropped_neymar'), Glob('cropped_messi'))
dataloader = DataLoader(dataset, batch_size=32, collate_fn=dataset.collate_fn)

In [5]:
# bloco de convolução
def _convLayer(input_features, output_features):
    return nn.Sequential(
        nn.Conv2d(input_features, output_features, kernel_size=5, stride=2, padding=2),
        nn.LeakyReLU(0.1, inplace=True)
    )

# bloco de upscale
def _UpScale(input_features, output_features):
    return nn.Sequential(
        nn.ConvTranspose2d(input_features, output_features, kernel_size=2, stride=2, padding=0),
        nn.LeakyReLU(0.1, inplace=True)
    )

# bloco que redimensiona a saída do encoder para o decoder
class Reshape(nn.Module):
    def forward(self, input):
        output = input.view(-1, 1024, 4, 4)
        return output

In [6]:
# criando a rede neural do deep fake
class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()

        # encoder para pegar as caracteristicas do rosto
        self.encoder = nn.Sequential(
            _convLayer(3, 128),
            _convLayer(128, 256),
            _convLayer(256, 512),
            _convLayer(512, 1024),
            nn.Flatten(),
            nn.Linear(1024 * 4 * 4, 1024),
            nn.Linear(1024, 1024 * 4 * 4),
            Reshape(),
            _UpScale(1024, 512),
        )

        # upscale diferente para os dois rostos (cada rosto, pesos diferentes)
        # upscale do rosto A
        self.decoder_A = nn.Sequential(
            _UpScale(512, 256),
            _UpScale(256, 128),
            _UpScale(128, 64),
            nn.Conv2d(64, 3, kernel_size=3, padding=1),
            nn.Sigmoid(),
        )

        # upscale do rosto B
        self.decoder_B = nn.Sequential(
            _UpScale(512, 256),
            _UpScale(256, 128),
            _UpScale(128, 64),
            nn.Conv2d(64, 3, kernel_size=3, padding=1),
            nn.Sigmoid(),
        )
    
    def forward(self, x, select='A'):
        if select == 'A':
            out = self.encoder(x)
            out = self.decoder_A(out)
        else:
            out = self.encoder(x)
            out = self.decoder_B(out)
        return out

In [7]:
from torchvision.utils import save_image

# instanciando o modelo
model = Autoencoder().to(device)
criterion = nn.L1Loss()

# instannciando os otimizadores para cada rosto
optimizer_A = optim.Adam(list(model.encoder.parameters()) + list(model.decoder_A.parameters()), lr=5e-5)
optimizer_B = optim.Adam(list(model.encoder.parameters()) + list(model.decoder_B.parameters()), lr=5e-5)

# quantidade de épocas
epochs = 200

# treinamento
for epoch in range(epochs):
    for i, (imsA, imsB, targetA, targetB) in enumerate(dataloader):
        
        # zera o gradiente do otimizador
        optimizer_A.zero_grad()
        optimizer_B.zero_grad()
        
        # treina o rosto A
        outA = model(imsA, select='A')
        lossA = criterion(outA, targetA)
        lossA.backward()
        optimizer_A.step()
        
        # treina o rosto B
        outB = model(imsB, select='B')
        lossB = criterion(outB, targetB)
        lossB.backward()
        optimizer_B.step()
    
    if epoch % 50 == 0:
        print(f'Epoch [{epoch}/{epochs}], Loss A: {lossA.item():.4f}, Loss B: {lossB.item():.4f}')

# termina o treinamento
model.eval()
with torch.no_grad():
    imsA, imsB, targetA, targetB = next(iter(dataloader))
    # troca o rosto (passa a imagem do rosto B no decoder do rosto A
    fake_neymar = model(imsB, select='A')
    save_image(fake_neymar[0], 'fake_neymar_on_messi.jpg')

Epoch [0/200], Loss A: 0.2004, Loss B: 0.2160
Epoch [50/200], Loss A: 0.1520, Loss B: 0.1469
Epoch [100/200], Loss A: 0.1189, Loss B: 0.1229
Epoch [150/200], Loss A: 0.0975, Loss B: 0.1098


In [ ]:
from torchvision import transforms as T
from torch.nn import functional as F
from torchvision.models import vgg19
import warnings
warnings.filterwarnings('ignore')

# baixa uma pintura do picasso
if not os.path.exists('Pratica/Fall_2022_web-images_Picasso_32.jpg'):
    !wget -q https://www.neh.gov/sites/default/files/2022-09/Fall_2022_web-images_Picasso_32.jpg

# preprocessamento
preprocess = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    T.Lambda(lambda x: x.mul_(255))
])

# pos processamento
postprocess = T.Compose([
    T.Lambda(lambda x: x.mul_(1./255)),
    T.Normalize(mean=[-0.485/0.229, -0.456/0.224,-0.406/0.225], std=[1/0.229,1/0.224,1/0.255]),
])

In [ ]:
# matrix
class GramMatrix(nn.Module):
    def forward(self,input):
        b,c,h,w = input.size()
        feat = input.view(b,c,h*w)
        G = feat@feat.transpose(1,2)
        G.div_(h*w)
        return G

class GramMSELoss(nn.Module):
    def forward(self,input,target):
        out = F.mse_loss(GramMatrix()(input),target)
        return(out)

class vgg19_modified(nn.Module):
    def __init__(self):
        super().__init__()
        features = list(vgg19(pretrained=True).features)
        self.features = nn.ModuleList(features).eval()

    def forward(self, x, layers=[]):
        order = np.argsort(layers)
        _results, results = [], []
        for ix, model in enumerate(self.features):
            x = model(x)
            if ix in layers:
                _results.append(x)
        for o in order:
            results.append(_results[o])
        return results if layers is not [] else x

vgg = vgg19_modified().to(device)

# Carrega as imagens para a transferência de estilo
imgs = [Image.open(path).resize((512,512)).convert('RGB') for path in ['Fall_2022_web-images_Picasso_32.jpg', 'fake_neymar_on_messi.jpg']]
style_image, content_image = [preprocess(img).to(device)[None] for img in imgs]

opt_img = content_image.data.clone()
opt_img.requires_grad = True

style_layers = [0, 5, 10, 19, 28]
content_layers = [21]
loss_layers = style_layers + content_layers

loss_fns = [GramMSELoss()] * len(style_layers) + [nn.MSELoss()] * len(content_layers)
loss_fns = [loss_fn.to(device) for loss_fn in loss_fns]

style_weights = [1000/n**2 for n in [64,128,256,512,512]]
content_weights = [1]
weights = style_weights + content_weights

style_targets = [GramMatrix()(A).detach() for A in vgg(style_image, style_layers)]
content_targets = [A.detach() for A in vgg(content_image, content_layers)]
targets = style_targets + content_targets

max_iters = 300 # Ajuste o LBFGS para processar mais rápido/devagar
optimizer = optim.LBFGS([opt_img])
log = Report(max_iters)

print("Iniciando Style Transfer...")
iters = 0
while iters < max_iters:
    def closure():
        global iters
        iters += 1
        optimizer.zero_grad()
        out = vgg(opt_img, loss_layers)
        layer_losses = [weights[a] * loss_fns[a](A, targets[a]) for a, A in enumerate(out)]
        loss = sum(layer_losses)
        loss.backward()
        log.record(pos=iters, loss=loss, end='\r')
        return loss
    optimizer.step(closure)

# Renderiza a imagem final
with torch.no_grad():
    out_img = postprocess(opt_img[0]).permute(1,2,0)
show(out_img)